# 04 — XGBoost — Clasificación Multiclase de Incontinencia Urinaria

## Objetivo

Entrenar y optimizar un clasificador XGBoost para predecir el tipo de incontinencia urinaria femenina (`mixed`, `none`, `stress`, `urge`) a partir de variables demográficas, clínicas y síntomas indirectos.

## Contexto: data leakage detectado

Durante el desarrollo inicial se identificó que las variables `ui_esfuerzo_presente` y `ui_urgencia_presente` tienen una asociación perfecta (Cramér's V = 1.0) con el target, porque el target fue construido directamente a partir de ellas. Entrenar con esas variables producía un F1-macro de 0.94, que caía a 0.46 al retirarlas — confirmando que el modelo aprendía una regla de codificación, no un patrón clínico real.

Por esta razón **ambas variables fueron eliminadas en el pipeline de preprocesamiento** y no están presentes en los CSVs que usa este notebook. El feature set resultante incluye variables demográficas, clínicas y síntomas indirectos (frecuencia, cantidad, molestia percibida, impacto en actividades) que describen la severidad de la incontinencia sin revelar su tipo directamente.

## Estructura del notebook

| Sección | Contenido |
|---|---|
| **1. Setup** | Librerías, datos, encoding del target y configuración de CV |
| **2. Baseline con CV** | ImbPipeline con SMOTE, validación cruzada estratificada 5-fold |
| **3. Tuning con Optuna** | Optimización de hiperparámetros, métrica objetivo F1-macro |
| **4. Verificación de overfitting** | Gap train-val ≤ 5% |
| **5. Métricas finales** | Accuracy, F1-macro, precision, recall, matriz de confusión |
| **6. Persistencia** | Guardado del modelo tuneado en `models/xgboost.pkl` |

---
## Sección 1 — Setup

Importamos todas las librerías necesarias y cargamos los datos del pipeline de preprocesamiento. Los CSVs ya no contienen las variables leaky — fueron eliminadas antes de este punto.

### Encoding del target

XGBoost requiere que las etiquetas de clase sean enteros consecutivos (0, 1, 2, 3) — no acepta strings directamente. Creamos un mapeo en orden alfabético y lo aplicamos a `y_train` e `y_test`:

- `mixed`  → 0
- `none`   → 1
- `stress` → 2
- `urge`   → 3

Este mapeo se aplica solo a las variables de target. Las features (`X_train`, `X_test`) no se modifican.

### Configuración de CV

La configuración de validación cruzada se carga del objeto compartido `cv_config.pkl` para garantizar que todos los modelos del proyecto usen exactamente los mismos folds.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import optuna
import joblib
import os
import warnings
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, accuracy_score, precision_score,
                             recall_score, ConfusionMatrixDisplay)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

print(f"XGBoost version: {xgb.__version__}")

In [ ]:
# Datos del pipeline (variables leaky ya eliminadas en preprocesamiento)
X_train = pd.read_csv('../data/processed/X_train_scaled.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
X_test  = pd.read_csv('../data/processed/X_test_scaled.csv')
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()

# Configuración de CV compartida con el resto de modelos del proyecto
cv_config = joblib.load('../models/cv_config.pkl')
skf = StratifiedKFold(**cv_config)

# XGBoost requiere etiquetas numéricas — mapeamos strings a 0,1,2,3
CLASS_ORDER = sorted(y_train.unique())
label_map   = {cls: i for i, cls in enumerate(CLASS_ORDER)}
y_train_enc = y_train.map(label_map).astype(int)
y_test_enc  = y_test.map(label_map).astype(int)

print(f"X_train: {X_train.shape}  |  y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}   |  y_test:  {y_test.shape}")
print(f"\nMapeo de clases: {label_map}")
print()
print("Distribución en train (sin SMOTE — distribución real):")
print((y_train.value_counts(normalize=True) * 100).round(1))

---
## Sección 2 — Baseline con validación cruzada estratificada

Antes de optimizar, establecemos una línea base de rendimiento usando los hiperparámetros por defecto de XGBoost.

### ImbPipeline con SMOTE

Usamos `ImbPipeline` para que SMOTE se aplique **dentro de cada fold** de la validación cruzada, no antes. Esto es crítico para evitar data leakage: si aplicáramos SMOTE fuera del fold, las muestras sintéticas generadas a partir de todo el conjunto de entrenamiento contaminarían el fold de validación, produciendo métricas artificialmente optimistas.

El flujo dentro de cada fold es:
1. Datos de entrenamiento del fold (reales) → SMOTE → datos balanceados
2. XGBoost entrena con los datos balanceados
3. Evalúa sobre el fold de validación (datos reales, sin sintéticos)

### scale_pos_weight

XGBoost tiene el parámetro `scale_pos_weight` para manejar el desbalanceo de clases en clasificación binaria. En clasificación multiclase con `multi:softprob` este parámetro no se aplica por clase directamente, por lo que el desbalanceo se gestiona a través de SMOTE en el pipeline.

In [ ]:
# Calculamos la proporción de desbalanceo para referencia
class_counts = y_train.value_counts()
total = len(y_train)

print("Distribución de clases y ratio de desbalanceo:")
print(f"{'Clase':<10} {'N':>6}  {'%':>6}  {'Ratio neg/pos':>14}")
print("-" * 45)
for cls in CLASS_ORDER:
    pos = class_counts[cls]
    neg = total - pos
    ratio = neg / pos
    print(f"{cls:<10} {pos:>6,}  {pos/total*100:>5.1f}%  {ratio:>14.2f}x")

print()
print("El desbalanceo se corrige con SMOTE dentro de cada fold del pipeline.")

In [ ]:
# Pipeline baseline: SMOTE + XGBoost con parámetros por defecto
pipeline_baseline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('xgb',   xgb.XGBClassifier(
        objective='multi:softprob',
        num_class=4,
        eval_metric='mlogloss',
        random_state=42,
        n_jobs=-1
    ))
])

cv_results_baseline = cross_validate(
    pipeline_baseline,
    X_train, y_train_enc,
    cv=skf,
    scoring={
        'accuracy':    'accuracy',
        'f1_macro':    'f1_macro',
        'f1_weighted': 'f1_weighted'
    },
    return_train_score=True,
    n_jobs=-1
)

baseline_f1_macro = cv_results_baseline['test_f1_macro'].mean()
baseline_accuracy = cv_results_baseline['test_accuracy'].mean()
baseline_gap      = (cv_results_baseline['train_f1_macro'].mean()
                     - cv_results_baseline['test_f1_macro'].mean())

print('=' * 55)
print('BASELINE — Validación cruzada 5-fold')
print('=' * 55)
print(f"Accuracy:      {baseline_accuracy:.4f} +/- {cv_results_baseline['test_accuracy'].std():.4f}")
print(f"F1-macro:      {baseline_f1_macro:.4f} +/- {cv_results_baseline['test_f1_macro'].std():.4f}")
print(f"F1-weighted:   {cv_results_baseline['test_f1_weighted'].mean():.4f} +/- {cv_results_baseline['test_f1_weighted'].std():.4f}")
print(f"Gap train-val: {baseline_gap:.4f}")

---
## Sección 3 — Optimización de hiperparámetros con Optuna

Usamos Optuna con el algoritmo **TPE (Tree-structured Parzen Estimator)** para buscar los mejores hiperparámetros. TPE aprende de los trials anteriores y concentra la búsqueda en las regiones más prometedoras del espacio, siendo más eficiente que un GridSearch exhaustivo.

### Control de overfitting

Cada trial mide simultáneamente el F1 en entrenamiento y en validación usando `cross_validate` con `return_train_score=True`. Si el gap entre ambos supera el **5%**, el trial recibe una penalización de `-1.0` y queda descartado automáticamente. Esto garantiza que Optuna no solo maximice la métrica de validación, sino que también respete la restricción de generalización del proyecto.

### Hiperparámetros buscados

| Parámetro | Rango | Escala | Qué controla |
|---|---|---|---|
| `n_estimators` | 100–800 | lineal | Número de árboles |
| `max_depth` | 3–8 | lineal | Profundidad máxima por árbol |
| `learning_rate` | 0.01–0.3 | log | Tasa de aprendizaje |
| `subsample` | 0.6–1.0 | lineal | Fracción de filas por árbol |
| `colsample_bytree` | 0.5–1.0 | lineal | Fracción de columnas por árbol |
| `min_child_weight` | 1–10 | lineal | Mínimo de muestras en hoja |
| `gamma` | 0–5 | lineal | Ganancia mínima para hacer un split |
| `reg_alpha` | 1e-8–1.0 | log | Regularización L1 (lasso) |
| `reg_lambda` | 1e-8–10.0 | log | Regularización L2 (ridge) |

In [ ]:
def objective(trial):
    params = {
        'objective':         'multi:softprob',
        'num_class':         4,
        'eval_metric':       'mlogloss',
        'random_state':      42,
        'n_jobs':            -1,
        'n_estimators':      trial.suggest_int('n_estimators', 100, 800),
        'max_depth':         trial.suggest_int('max_depth', 3, 8),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
        'gamma':             trial.suggest_float('gamma', 0, 5),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }

    pipeline = ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('xgb',   xgb.XGBClassifier(**params)),
    ])

    cv_results = cross_validate(
        pipeline, X_train, y_train_enc,
        cv=skf,
        scoring='f1_macro',
        return_train_score=True,
        n_jobs=-1
    )

    f1_val   = cv_results['test_score'].mean()
    f1_train = cv_results['train_score'].mean()
    gap      = f1_train - f1_val

    # Penalizar si el overfitting supera el 5%
    if gap > 0.05:
        return -1.0

    return f1_val

In [ ]:
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name='XGB_Optimization'
)
study.optimize(objective, n_trials=100, show_progress_bar=True)

valid_trials = [t for t in study.trials if t.value is not None and t.value > 0]

print(f"\nTrials totales:  {len(study.trials)}")
print(f"Trials válidos:  {len(valid_trials)}  (pasaron filtro overfitting ≤ 5%)")
print(f"\nMejor F1-val CV: {study.best_value:.4f}")
print("Mejores parámetros:")
for k, v in study.best_params.items():
    print(f"  {k:<22}: {v}")

---
## Sección 4 — Verificación del overfitting

Entrenamos el modelo con los mejores hiperparámetros encontrados por Optuna y medimos explícitamente tres niveles de rendimiento:

- **F1 Train (CV):** rendimiento promedio en los folds de entrenamiento
- **F1 Val (CV):** rendimiento promedio en los folds de validación — estimador del rendimiento real
- **F1 Test:** rendimiento sobre el conjunto de test, que no participó en ningún momento del tuning
- **Gap train-val:** debe ser ≤ 0.05 (restricción del proyecto)
- **Gap val-test:** detecta si hubo optimismo excesivo durante la búsqueda de Optuna

In [ ]:
best_model = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('xgb',   xgb.XGBClassifier(
        objective='multi:softprob',
        num_class=4,
        eval_metric='mlogloss',
        random_state=42,
        n_jobs=-1,
        **study.best_params
    )),
])

cv_final = cross_validate(
    best_model, X_train, y_train_enc,
    cv=skf, scoring='f1_macro',
    return_train_score=True
)

f1_train_final = cv_final['train_score'].mean()
f1_val_final   = cv_final['test_score'].mean()
gap_final      = f1_train_final - f1_val_final

# Entrenamiento final sobre todo el train y evaluación en test
best_model.fit(X_train, y_train_enc)
y_pred_test = best_model.predict(X_test)
f1_test     = f1_score(y_test_enc, y_pred_test, average='macro')

print("=" * 45)
print("VERIFICACIÓN DE OVERFITTING — MODELO FINAL")
print("=" * 45)
print(f"F1 Train (CV):   {f1_train_final:.4f}")
print(f"F1 Val   (CV):   {f1_val_final:.4f}")
print(f"F1 Test:         {f1_test:.4f}")
print(f"Gap train-val:   {gap_final:.4f}  {'✓ OK' if gap_final <= 0.05 else '✗ EXCEDE 5%'}")
print(f"Gap val-test:    {abs(f1_val_final - f1_test):.4f}  {'✓ OK' if abs(f1_val_final - f1_test) <= 0.05 else '✗ revisar'}")

---
## Sección 5 — Métricas finales

Comparamos el baseline contra el modelo tuneado y calculamos todas las métricas requeridas: accuracy, F1-macro, precision y recall por clase, y matriz de confusión.

La mejora en `urge` y `stress` es el indicador más relevante — son las clases con menor señal disponible sin las variables leaky. `none` es la clase mayoritaria y suele tener el mejor rendimiento en todos los modelos.

In [ ]:
# Baseline entrenado sobre todo el train para comparación en test
pipeline_baseline.fit(X_train, y_train_enc)
y_pred_base = pipeline_baseline.predict(X_test)

print("BASELINE (parámetros por defecto):")
print(classification_report(y_test_enc, y_pred_base, target_names=CLASS_ORDER))

print("MODELO TUNED (Optuna 100 trials):")
print(classification_report(y_test_enc, y_pred_test, target_names=CLASS_ORDER))

# Mejora por clase
f1_base_cls  = f1_score(y_test_enc, y_pred_base,  average=None)
f1_tuned_cls = f1_score(y_test_enc, y_pred_test, average=None)

print("MEJORA POR CLASE (baseline → tuneado):")
for cls, base, tuned in zip(CLASS_ORDER, f1_base_cls, f1_tuned_cls):
    delta = tuned - base
    arrow = '↑' if delta > 0 else '↓'
    print(f"  {cls:<10}  {base:.3f} → {tuned:.3f}  {arrow} {abs(delta):.3f}")

In [ ]:
# Métricas globales del modelo tuneado
print("=" * 45)
print("MÉTRICAS GLOBALES — MODELO TUNEADO")
print("=" * 45)
print(f"Accuracy:        {accuracy_score(y_test_enc, y_pred_test):.4f}")
print(f"F1-macro:        {f1_score(y_test_enc, y_pred_test, average='macro'):.4f}")
print(f"Precision-macro: {precision_score(y_test_enc, y_pred_test, average='macro'):.4f}")
print(f"Recall-macro:    {recall_score(y_test_enc, y_pred_test, average='macro'):.4f}")
print(f"Gap train-val:   {gap_final:.4f}  {'✓ ≤ 5%' if gap_final <= 0.05 else '✗ > 5%'}")

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test_enc, y_pred_test)
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_ORDER)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Matriz de confusión — XGBoost tuneado', fontweight='bold')
plt.tight_layout()
os.makedirs('../assets', exist_ok=True)
plt.savefig('../assets/14_confusion_matrix_xgboost.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Sección 6 — Persistencia del modelo

Se guarda únicamente el modelo con mejor rendimiento en validación cruzada: el pipeline tuneado con Optuna. Este archivo es el único artefacto que consume el ensemble.

El objeto guardado es un `ImbPipeline` que encapsula tanto el paso de SMOTE como el clasificador XGBoost con los hiperparámetros óptimos. Al cargarlo para inferencia, el paso de SMOTE se omite automáticamente — solo se ejecuta durante el entrenamiento.

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/xgboost.pkl')

print("Modelo guardado: ../models/xgboost.pkl")
print(f"  Tipo:           ImbPipeline (SMOTE + XGBClassifier)")
print(f"  F1-macro test:  {f1_test:.4f}")
print(f"  Gap train-val:  {gap_final:.4f}  {'✓ ≤ 5%' if gap_final <= 0.05 else '✗ > 5%'}")